# 03 · La convolución como matriz
*im2col, pesos compartidos y equivarianza a la traslación*

Notebook del episodio 3 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/la-convolucion-como-matriz)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** Toeplitz 4×6 con w = [1, 0, −1]: T@x = conv1d = [1, −1, −1, 1] → F.unfold de una imagen 1..16 con k=2: (1, 4, 9), el 6 en 4 columnas → im2col: cols (1, 27, 1024), W (16, 27), dif. con conv2d 0 (cuDNN, Chetlur et al. 2014) → parámetros: conv 3→16 k3 = 448 vs. Linear(3072, 16384) en device meta = 50.348.032 (112.384×) → equivarianza en la flor 26×40 con roll de 6 px: dif. 0 en el interior, solo columnas 0, 5, 6, 39 → coseno tras correr 1 px un mapa de bordes: 0,65 sin pooling, 0,82 / 0,93 / 0,98 con max pool 2/4/8

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio tres: la convolución como matriz. En el episodio anterior la escribimos como un kernel que se desliza. Hoy la vamos a escribir como una multiplicación de matrices, que es como la calculan de verdad las placas de video, y de esa forma van a salir solas sus dos propiedades más importantes.*

### Qué vamos a ver

- En 1D, una convolución es una matriz con diagonales repetidas
- En 2D, unfold arma columnas de parches: una matmul
- Pesos compartidos: 448 parámetros contra 50 millones
- Mover la entrada mueve la salida; el pooling lo suaviza

El plan. Primero, en una dimensión: una convolución es multiplicar por una matriz muy particular, con el mismo kernel repetido en cada fila. En dos dimensiones, una función de PyTorch, unfold, recorta todos los parches y los pone como columnas: después alcanza una sola multiplicación de matrices. Esa forma deja ver cuánto ahorra compartir pesos: cuatrocientos cuarenta y ocho parámetros contra cincuenta millones. Y por último, la equivarianza: si movés la imagen, la salida se mueve igual. Lo medimos, y vemos qué agrega el pooling.

## Parte 1 · Una convolución 1D es una matriz

Empecemos con una señal de seis números y un kernel de tres. Todo entra en pantalla.

### Armar la matriz a mano

El código. x es una señal de seis valores, como una fila de píxeles. w es el kernel: uno, cero, menos uno. Armamos una matriz T de ceros, de cuatro por seis: cuatro filas porque el kernel de tres entra en cuatro posiciones de seis. Y en cada fila copiamos el mismo w, corrido un lugar más a la derecha que en la fila anterior. Comparamos T por x con la convolución de una dimensión de PyTorch, F punto conv1d. La matriz impresa tiene forma de escalera, y los dos resultados son idénticos: uno, menos uno, menos uno, uno.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_toeplitz.py
__file__ = 'snippets/ep03_toeplitz.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

torch.set_printoptions(precision=0, sci_mode=False)

In [3]:
x = torch.tensor([1., 2., 0., 3., 1., 2.])   # señal de 6 valores
w = torch.tensor([1., 0., -1.])              # kernel de 3
T = torch.zeros(4, 6)                        # 4 posiciones × 6 valores
for i in range(4):
    T[i, i:i + 3] = w                        # el mismo w, corrido
print(T)
print("T @ x  :", (T @ x).tolist())
print("conv1d :", F.conv1d(x[None, None], w[None, None])[0, 0].tolist())

tensor([[ 1.,  0., -1.,  0.,  0.,  0.],
        [ 0.,  1.,  0., -1.,  0.,  0.],
        [ 0.,  0.,  1.,  0., -1.,  0.],
        [ 0.,  0.,  0.,  1.,  0., -1.]])
T @ x  : [1.0, -1.0, -1.0, 1.0]
conv1d : [1.0, -1.0, -1.0, 1.0]


In [4]:
# Verificación extra (corrió junto con el video)
json.dump({"T": T.int().tolist()}, open(ruta_lab("datos", "figs", "ep03_toeplitz.json"), "w"))

### La matriz T: el mismo kernel en cada fila

Mirala como imagen. En azul los unos, en rojo los menos uno, y todo lo demás vale cero. Cada fila es una posición del kernel: la fila cero mira x cero a x dos, la fila uno mira x uno a x tres, y así. Fijate en las diagonales: cada una tiene siempre el mismo número. A una matriz así se le dice de Toeplitz. De veinticuatro casillas, solo hay tres números distintos, los tres del kernel.

### Convolución = matriz por vector

$$
y = T\,x,\quad T \in \mathbb{R}^{4\times 6},\ x \in \mathbb{R}^{6}
$$
<small>cada salida: una fila de T por x</small>

$$
T_{ij} = \begin{cases} w_{\,j-i} & 0 \le j-i < k \\ 0 & \text{si no} \end{cases}
$$
<small>el valor depende solo de j − i: la diagonal</small>

En símbolos. y es igual a T por x: T es una matriz de cuatro por seis, x un vector de seis números, y sale un vector de cuatro. En el dibujo, la fila uno de T recorre x y llena la celda uno de y: dos por uno, más tres por menos uno, da menos uno. Es exactamente la ventana del kernel sobre x uno, x dos, x tres. Y la regla de T: la casilla de la fila i, columna j, vale w sub j menos i si esa resta cae entre cero y k menos uno, y cero si no. Como depende solo de la resta, toda la diagonal tiene el mismo valor. Eso es compartir pesos, escrito como matriz.

## Parte 2 · im2col: parches en columnas

En dos dimensiones la matriz de Toeplitz se vuelve enorme. Las librerías hacen otra cosa: dan vuelta el problema.

### El truco im2col

El truco se llama im2col: de imagen a columnas. Tenemos una imagen de tres canales de treinta y dos por treinta y dos. Recortamos todos los parches de tres por tres, con sus tres canales. Cada parche son veintisiete números, y los ponemos como una columna: mil veinticuatro columnas, una por posición. Los dieciséis kernels también se aplanan, en una matriz de dieciséis por veintisiete. Una sola multiplicación de matrices hace todas las sumas de productos juntas. Y la salida se reacomoda como dieciséis canales de treinta y dos por treinta y dos.

imagen 3×32×32 → recortar parches → columnas 27×1024 → kernels 16×27 → una matmul → salida 16×32×32

### F.unfold, con números que se reconocen

Veamos qué hace unfold con una imagen fácil de seguir. Una imagen de cuatro por cuatro con los números del uno al dieciséis. F punto unfold con parches de dos por dos. La forma de la salida: batch uno, cuatro valores por parche, y nueve parches, porque un kernel de dos entra en tres posiciones por lado. Leé la primera columna de arriba abajo: uno, dos, cinco, seis. Es el parche de arriba a la izquierda. La segunda columna, dos, tres, seis, siete: el parche corrido un lugar.

In [5]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_unfold.py
__file__ = 'snippets/ep03_unfold.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

In [6]:
x = torch.arange(1., 17.).view(1, 1, 4, 4)   # imagen 4 × 4 con 1..16
cols = F.unfold(x, kernel_size=2)             # parches de 2 × 2
print("cols:", tuple(cols.shape))             # (batch, 4 valores, 9 parches)
print(cols[0].long())

cols: (1, 4, 9)
tensor([[ 1,  2,  3,  5,  6,  7,  9, 10, 11],
        [ 2,  3,  4,  6,  7,  8, 10, 11, 12],
        [ 5,  6,  7,  9, 10, 11, 13, 14, 15],
        [ 6,  7,  8, 10, 11, 12, 14, 15, 16]])


In [7]:
# Verificación extra (corrió junto con el video)
json.dump({"x": x[0, 0].int().tolist(), "cols": cols[0].int().tolist()}, open(ruta_lab("datos", "figs", "ep03_unfold.json"), "w"))

### Cada columna es un parche

La misma matriz, como imagen. Cada columna, de p cero a p ocho, es un parche; cada fila es una de las cuatro posiciones dentro del parche. Buscá el seis: aparece cuatro veces, porque ese píxel cae en cuatro parches distintos. Ese es el precio de im2col: copia los datos. Con kernels de tres por tres, cada píxel se repite hasta nueve veces.

### La convolución como una sola matmul

Ahora con una imagen de verdad en tamaño: tres canales de treinta y dos por treinta y dos, y dieciséis kernels de tres por tres por tres. Unfold, con padding uno, da la matriz de columnas: veintisiete filas, que son C in, los canales de entrada, por k al cuadrado, y mil veinticuatro columnas, una por píxel de salida. Los kernels, aplanados con view: dieciséis filas de veintisiete. Una multiplicación, y view para volver a la forma de imagen. Y comparamos con F punto conv2d. La diferencia es cero exacto: las dos hacen las mismas sumas en el mismo orden.

**Cada línea, su término:**

- `cols = F.unfold` → $X_{\text{col}} \in \mathbb{R}^{27 \times 1024}$ — $C_{in}\,k^2$ filas, una columna por posición
- `W = w.view(16, -1)` → $W_{\text{mat}} \in \mathbb{R}^{16 \times 27}$ — un kernel aplanado por fila
- `y = (W @ cols)` → $Y = W_{\text{mat}}\, X_{\text{col}} \in \mathbb{R}^{16 \times 1024}$ — todas las sumas de productos juntas
- `ref = F.conv2d` → $Y \equiv \mathrm{conv2d}(x, w)$ — mismo resultado

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_im2col.py
__file__ = 'snippets/ep03_im2col.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F

torch.manual_seed(0)

In [9]:
x = torch.randn(1, 3, 32, 32)                 # una imagen de 3 canales
w = torch.randn(16, 3, 3, 3)                  # 16 kernels de 3 × 3 × 3
cols = F.unfold(x, kernel_size=3, padding=1)  # (1, 27, 1024)
W = w.view(16, -1)                            # (16, 27)
y = (W @ cols).view(1, 16, 32, 32)            # una matmul y reacomodar
ref = F.conv2d(x, w, padding=1)
print("cols", tuple(cols.shape), " W", tuple(W.shape))
print("y   ", tuple(y.shape))
print("diferencia con conv2d:", f"{(y - ref).abs().max():.1e}")

cols (1, 27, 1024)  W (16, 27)
y    (1, 16, 32, 32)
diferencia con conv2d: 0.0e+00


### Una celda de la salida

$$
Y_{o,p} = \sum_{q=1}^{27} W_{o,q}\, X_{q,p}
$$
<small>kernel o, parche p</small>

$$
27 \times 1024 = 27\,648 \;\text{vs.}\; 3 \times 1024 = 3072
$$
<small>la matriz de columnas ocupa 9 veces más que la imagen</small>

Leamos una celda. Y sub o, p es la suma, sobre las veintisiete posiciones q, de W sub o q por X sub q p. o elige el kernel, o sea el canal de salida, y p elige la posición en la imagen. En el dibujo: la fila o de los kernels recorre la columna p de los parches, término a término, y llena una sola celda. Es la misma suma de productos del episodio anterior, nada más que ordenada para que la haga una rutina de multiplicación de matrices, que está optimizadísima. El costo: la matriz de columnas tiene veintisiete mil seiscientos cuarenta y ocho números, nueve veces lo que ocupa la imagen.

> **Chetlur et al., 2014 — cuDNN: Efficient Primitives for Deep Learning** · [arXiv:1410.0759](https://arxiv.org/abs/1410.0759)
>
> **K × CRS · CRS × NPQ** · «One approach is to lower the convolutions into a matrix multiplication»: filtros como matriz K × CRS y datos duplicados en una CRS × NPQ.

Esto no es solo didáctico. El paper de cuDNN, la librería de NVIDIA que usan casi todos los frameworks en placas de video, lo describe así: bajar la convolución a una multiplicación de matrices, con los filtros como una matriz de K por C R S y los datos duplicados en otra. Es nuestra W mat y nuestra X col. Y cuentan el problema que vimos: materializar esa matriz gasta memoria, así que la arman de a pedazos dentro del chip.

## Parte 3 · Pesos compartidos

Ahora comparemos con lo que haría una capa densa que mapee la misma entrada a la misma salida.

### Contar parámetros

Las dos capas transforman lo mismo: una imagen de tres por treinta y dos por treinta y dos en dieciséis canales del mismo tamaño. La convolución de tres por tres. Y una capa lineal que conecte cada uno de los tres mil setenta y dos números de entrada con cada uno de los dieciséis mil trescientos ochenta y cuatro de salida. La creamos en el device meta, que guarda solo las formas y no reserva memoria. Contamos los números de cada parámetro. La convolución tiene cuatrocientos cuarenta y ocho. La densa, cincuenta millones trescientos cuarenta y ocho mil treinta y dos: ciento doce mil veces más.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_parametros.py
__file__ = 'snippets/ep03_parametros.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch.nn as nn

In [11]:
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)    # 3×32×32 → 16×32×32
densa = nn.Linear(3 * 32 * 32, 16 * 32 * 32, device="meta")  # mismo mapeo
n_conv = sum(p.numel() for p in conv.parameters())
n_densa = sum(p.numel() for p in densa.parameters())
print(f"conv : {n_conv:>12,}")
print(f"densa: {n_densa:>12,}")
print(f"la densa tiene {n_densa / n_conv:,.0f} veces más")

conv :          448
densa:   50,348,032
la densa tiene 112,384 veces más


### De dónde salen esos números

$$
C_{out}\,(C_{in}\,k^2 + 1) = 16\,(27 + 1) = 448
$$
<small>pesos de los kernels + un bias por canal</small>

$$
(3\cdot 32^2)(16\cdot 32^2) + 16\cdot 32^2 = 50\,348\,032
$$
<small>cada entrada con cada salida</small>

La cuenta. La convolución tiene C out kernels, cada uno con C in por k al cuadrado pesos, más un bias por kernel: dieciséis por veintiocho, cuatrocientos cuarenta y ocho. En el dibujo, cuatrocientos treinta y dos pesos más dieciséis bias. Fijate que no aparece el tamaño de la imagen: la misma capa sirve para una foto de treinta y dos o de mil píxeles. La densa conecta todo con todo: tres mil setenta y dos por dieciséis mil trescientos ochenta y cuatro, más los bias. Y si la imagen duplica el lado, eso se multiplica por dieciséis. La convolución es la densa con casi todo en cero y el resto repetido.

## Parte 4 · Equivarianza a la traslación

Compartir pesos tiene una consecuencia que se puede medir: el mismo kernel en todos lados reacciona igual en todos lados.

### Mover y filtrar, o filtrar y mover

Usamos la flor achicada del episodio anterior, de veintiséis por cuarenta. Un kernel cualquiera, al azar: la propiedad no depende de qué detecte. Mover es torch punto roll: correr la imagen seis píxeles a la derecha; lo que sale por un costado entra por el otro. Primero movemos y después filtramos. Y al revés: filtramos y después movemos. En el interior la diferencia es cero exacto. Solo hay diferencias en cuatro columnas, cerca de los costados, donde el roll dio la vuelta y el padding de ceros no coincide.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_equivarianza.py
__file__ = 'snippets/ep03_equivarianza.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

foto = torch.from_numpy(np.load(ruta_lab("datos", "fotos.npz"))["flor"]).float() / 255
gris = (foto @ torch.tensor([0.299, 0.587, 0.114]))[None, None]
chica = F.avg_pool2d(gris, 16)                        # la flor en 26 × 40
torch.manual_seed(0)

In [13]:
w = torch.randn(1, 1, 3, 3)                           # un kernel cualquiera
conv = lambda img: F.conv2d(img, w, padding=1)
mover = lambda img: torch.roll(img, shifts=6, dims=3) # 6 píxeles a la derecha

a = conv(mover(chica))                                # mover y después filtrar
b = mover(conv(chica))                                # filtrar y después mover
dif = (a - b).abs()[0, 0]
print("forma:", tuple(dif.shape))
print(f"diferencia en el interior   : {dif[:, 8:-1].max():.1e}")
print(f"diferencia en todo          : {dif.max():.2f}")
cols = torch.nonzero(dif.amax(0) > 1e-5).flatten().tolist()
print("columnas con diferencia:", cols)

forma: (26, 40)
diferencia en el interior   : 0.0e+00
diferencia en todo          : 1.13
columnas con diferencia: [0, 5, 6, 39]


In [14]:
# Verificación extra (corrió junto con el video)
json.dump({"dif": [[round(float(v), 3) for v in f] for f in dif]}, open(ruta_lab("datos", "figs", "ep03_equivarianza.json"), "w"))

### Dónde difieren los dos caminos

La diferencia como imagen. Todo oscuro, cero, salvo unas columnas cerca de los costados: donde la imagen dio la vuelta. En el resto de la foto, mover y filtrar conmutan. La flor entera, con sus bordes, se detecta igual estés donde estés.

### Equivarianza e invarianza

$$
f(T_\delta\, x) = T_\delta\, f(x)
$$
<small>equivarianza: $T_\delta$ corre la imagen δ píxeles</small>

$$
g(T_\delta\, x) \approx g(x)
$$
<small>invarianza: la salida casi no cambia</small>

Dos palabras parecidas que conviene separar. T sub delta es la operación de correr la imagen delta píxeles; en el dibujo, mover es solo elegir los mismos valores en otro orden. Equivarianza quiere decir que filtrar la imagen movida es lo mismo que mover la imagen filtrada: f y T sub delta conmutan. La convolución es equivariante. Invarianza es otra cosa: que g, una función que llega hasta la clase, directamente no cambie. Para decir gato no importa dónde está el gato. La convolución sola no lo da: el borde se movió con la flor. Hace falta algo más.

### El max pooling da invarianza aproximada

Ese algo es el pooling, que el episodio que viene vemos en detalle: quedarse con el máximo de cada bloque. Un mapa de bordes de la flor, en tamaño completo, sin los negativos. El mismo mapa corrido un solo píxel. Medimos cuánto se parecen con el coseno entre los dos, aplanados: uno es idénticos. Y repetimos después de max pooling de dos, cuatro y ocho. Sin pooling, un píxel de corrimiento baja el parecido a cero coma sesenta y cinco. Con pooling de dos sube a cero coma ochenta y dos, y con ocho, a cero coma noventa y ocho. Invarianza aproximada, a costa de perder resolución.

In [15]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep03_pooling.py
__file__ = 'snippets/ep03_pooling.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

foto = torch.from_numpy(np.load(ruta_lab("datos", "fotos.npz"))["flor"]).float() / 255
gris = (foto @ torch.tensor([0.299, 0.587, 0.114]))[None, None]
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])[None, None]

In [16]:
mapa = F.relu(F.conv2d(gris, sobel_x))        # bordes de la flor
movido = torch.roll(mapa, 1, dims=3)          # 1 píxel a la derecha
coseno = lambda a, b: F.cosine_similarity(a.flatten(), b.flatten(), 0)

print(f"sin pooling     : {coseno(mapa, movido):.2f}")
for k in (2, 4, 8):
    a, b = F.max_pool2d(mapa, k), F.max_pool2d(movido, k)
    print(f"max pooling {k}×{k} : {coseno(a, b):.2f}")

sin pooling     : 0.65
max pooling 2×2 : 0.82
max pooling 4×4 : 0.93
max pooling 8×8 : 0.98


## Resumiendo

- Conv 1D = matriz de Toeplitz: el mismo w en cada fila
- im2col: unfold arma columnas de parches + una matmul
- Y = W_mat · X_col, a costa de copiar cada píxel
- 448 parámetros contra 50 millones de una densa
- Equivariante: mover y filtrar conmutan; pooling ≈ invarianza
- Próximo: canales, convoluciones 1×1 y campo receptivo